In [ ]:
# Cell 0: PDF → images
import pypdfium2 as pdfium, os
PDF_PATH = os.path.expanduser('~/scratch/original_file.pdf')
SCALE = 200 / 72  # 200dpi is optimal for this document type
pdf = pdfium.PdfDocument(PDF_PATH)
images = [page.render(scale=SCALE).to_pil().convert('RGB') for page in pdf]
print(f'{len(images)} pages, size: {images[0].size}')


In [ ]:
# Cell 1: Surya OCR
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor
from surya.foundation import FoundationPredictor
det_predictor = DetectionPredictor()
rec_predictor = RecognitionPredictor(FoundationPredictor())
all_predictions = rec_predictor(images, det_predictor=det_predictor)
print('OCR done')


In [ ]:
# 3) Parse OCR output 
import pandas as pd
import re
import numpy as np

ROW_THRESHOLD = 8
DIRECTIONAL = re.compile(
    r'^(søndre|nordre|østre|vestre|øvre|nedre|lille|store|skole-?|mellem)\s+(.*)', re.I)
SOGN_RE = re.compile(r'\bsogn\.?\b', re.I)
HEADER_Y_LIMIT = 350
DASH_RE = re.compile(r'^[\s\u2014\u2013\u2012\u2015—–-]+$')


def detect_columns(pred):
    headers = {}
    for line in pred.text_lines:
        if line.bbox[1] > HEADER_Y_LIMIT: break
        t = line.text.strip().lower()
        x0 = line.bbox[0]
        if 'gaards' in t and 'gaards' not in headers: headers['gaards'] = x0
        elif 'brugs' in t and 'brugs' not in headers: headers['brugs'] = x0
        elif 'gaardens' in t and 'gaardens' not in headers: headers['gaardens'] = x0
        elif 'brugets' in t and 'brugets' not in headers: headers['brugets'] = x0
        elif ('eierens' in t or 'brugerens' in t) and 'eier' not in headers: headers['eier'] = x0
        elif t in ('mark.', 'mark'): headers['mark'] = x0
        elif t in ('ore.', 'øre.', 'ore', 'øre'): headers['ore'] = x0
        elif 'anmerkn' in t and 'anmerkn' not in headers: headers['anmerkn'] = x0
        elif 'postanstalt' in t and 'post' not in headers: headers['post'] = x0
    if len(headers) < 5: return None
    g = headers.get('gaards', 40); b = headers.get('brugs', g+35)
    gn = headers.get('gaardens', b+55); bn = headers.get('brugets', gn+200)
    ek = headers.get('eier', bn+200); mk = headers.get('mark', ek+200)
    ore = headers.get('ore', mk+40)
    anm = headers.get('anmerkn', ore+60)
    post = headers.get('post', anm+120)
    cols = [
        ('gaards_no',     g-15,       (g+b)/2),
        ('brugs_no',      (g+b)/2,    (b+gn)/2),
        ('gaardens_navn', (b+gn)/2,   (gn+bn)/2),
        ('brugets_navn',  (gn+bn)/2,  (bn+ek)/2),
        ('eier_bruker',   (bn+ek)/2,  (ek+mk)/2),
        ('skyld',         (ek+mk)/2,  (ore+anm)/2),
        ('anmerkn',       (ore+anm)/2, (anm+post)/2),
        ('postanstalt',   (anm+post)/2, post+200),
    ]
    bex = (bn+ek)/2
    header_y = max(l.bbox[3] for l in pred.text_lines
                   if l.bbox[1] < HEADER_Y_LIMIT and l.text.strip().lower() in
                   ('mark.','ore.','øre.','mark','ore','øre')) + 20
    return cols, bex, header_y


def col_at(x, columns):
    for i, (_, lo, hi) in enumerate(columns):
        if lo <= x < hi: return i
    return -1


def find_split(text, target, lo=None, hi=None):
    lo = max(1, lo if lo is not None else target-15)
    hi = min(len(text)-1, hi if hi is not None else target+15)
    dots = [j for j in range(lo, hi) if j+1 < len(text) and text[j]=='.' and text[j+1]=='.']
    if dots:
        best = min(dots, key=lambda j: abs(j-target))
        end = best
        while end < len(text) and text[end]=='.': end += 1
        return end
    caps = [j for j in range(lo, hi)
            if j+1 < len(text) and text[j+1].isupper() and text[j] in (' ','-')]
    if caps: return min(caps, key=lambda j: abs(j-target))
    spaces = [j for j in range(lo, hi) if text[j] == ' ']
    if spaces: return min(spaces, key=lambda j: abs(j-target))
    return None


def split_multi(bbox, text, columns, col_indices):
    if len(col_indices) < 2: return [(columns[col_indices[0]][0], text)]
    x0, x1 = bbox[0], bbox[2]
    w = x1 - x0
    if w < 1: return [(columns[col_indices[0]][0], text)]
    splits = []
    for k in range(len(col_indices)-1):
        bx = columns[col_indices[k]][2]
        ct = round(len(text) * (bx - x0) / w)
        slo = (splits[-1]+2) if splits else 1
        sp = find_split(text, ct, lo=slo)
        if sp is not None: splits.append(sp)
    if len(splits) != len(col_indices)-1:
        return [(columns[col_indices[0]][0], text)]
    result, start = [], 0
    for k, sp in enumerate(splits):
        part = text[start:sp+1] if text[sp] == '-' else text[start:sp]
        start = sp + 1
        part = re.sub(r'^[\s.]+|[\s.]+$', '', part).strip()
        if part: result.append((columns[col_indices[k]][0], part))
    part = re.sub(r'^[\s.]+|[\s.]+$', '', text[start:]).strip()
    if part: result.append((columns[col_indices[-1]][0], part))
    return result


def process_line(bbox, text, columns, bex, skyld_lo):
    x0, x1 = bbox[0], bbox[2]
    ci = col_at(x0, columns)
    if ci < 0: return []
    if ci <= 1 and re.match(r'^\d+\s*$', text.strip()):
        return [(columns[ci][0], text.strip())]
    if ci == 6 and len(columns) > 7:
        post_lo = columns[7][1]
        if x0 >= post_lo - 30:
            return [('postanstalt', text)]
        if x1 > post_lo - 20:
            w = x1 - x0
            if w > 1:
                ct = round(len(text) * (post_lo - x0) / w)
                sp = find_split(text, ct, lo=max(1, ct-15), hi=min(len(text)-1, ct+15))
                if sp:
                    return [('anmerkn', text[:sp].strip()), ('postanstalt', text[sp:].strip())]
        return [('anmerkn', text)]
    if ci >= 4: return [(columns[ci][0], text)]
    if ci == 3:
        if x1 > skyld_lo - 20: return split_multi(bbox, text, columns, [3, 4])
        return [('brugets_navn', text)]
    if ci == 2:
        if x1 > skyld_lo - 20: return split_multi(bbox, text, columns, [2, 3, 4])
        if x1 > bex: return split_multi(bbox, text, columns, [2, 3])
        return [('gaardens_navn', text)]
    if ci == 1:
        m = re.match(r'^(\d+)\s+(.+)', text.strip())
        if m:
            num, rest = m.group(1), m.group(2)
            rx0 = x0 + (len(num)+1) / max(1, len(text)) * (x1-x0)
            return [('brugs_no', num)] + process_line(
                [rx0, bbox[1], x1, bbox[3]], rest, columns, bex, skyld_lo)
        return [('brugs_no', text.strip())]
    return [('gaards_no', text.strip())]


def parse_skyld(text):
    s = text.strip().replace('\u201e', ',,').replace('\u201c', ',,').replace('|', ' ')
    anm = ''
    m_anm = re.search(r'\s+([A-Za-zÆØÅæøå][\w\s.\-æøåÆØÅ]*?)$', s)
    if m_anm: anm = m_anm.group(1).strip()
    s = re.sub(r'\s+[A-Za-zÆØÅæøå].*$', '', s).strip()
    if not s or not any(c.isdigit() for c in s): return '', '', anm
    if s.startswith(',,') or s.startswith('.,'):
        m = re.match(r'^(\d{1,2})', s[2:].strip())
        return (('', m.group(1).zfill(2), anm) if m else ('', s[2:].strip(), anm))
    m = re.match(r'^(\d+)\s+(\d{1,2})\b', s)
    if m: return m.group(1), m.group(2).zfill(2), anm
    # FIX: bare 1-2 digit number = ore only (,,XX lost by OCR)
    if re.match(r'^\d{1,2}$', s):
        return '', s.zfill(2), anm
    m = re.match(r'^(\d+?)(\d{2})$', s)
    if m: return m.group(1), m.group(2).zfill(2), anm
    return s, '', anm


def clean(text):
    text = re.sub(r'^[\s.]{2,}', '', text)
    text = re.sub(r'[\s.]*\.[\s.]*\.[\s.]*$', '', text).strip()
    text = re.sub(r'\s\.$', '', text).strip()
    text = text.replace('|', '').strip()
    return re.sub(r'-\s+(?=[a-zæøå])', '', text)


def clean_anmerkn(text):
    t = text.strip()
    if not t or DASH_RE.match(t): return ''
    return t


def is_skip(text):
    t = text.strip()
    return bool(SOGN_RE.search(t)) or bool(re.match(r'^[\s.]+$', t))


def cluster_rows(items):
    if not items: return []
    items.sort(key=lambda x: (x[0][1]+x[0][3])/2)
    clusters = [[items[0]]]
    for item in items[1:]:
        ym = (item[0][1]+item[0][3])/2
        pm = (clusters[-1][-1][0][1]+clusters[-1][-1][0][3])/2
        if abs(ym-pm) <= ROW_THRESHOLD: clusters[-1].append(item)
        else: clusters.append([item])
    final = []
    for cluster in clusters:
        seen, sub = set(), []
        for item in sorted(cluster, key=lambda x: (x[0][1]+x[0][3])/2):
            col = item[1]
            if col in seen and col in ('gaards_no','gaardens_navn','brugs_no','brugets_navn','eier_bruker'):
                final.append(sub); sub = [item]; seen = {col}
            else: sub.append(item); seen.add(col)
        if sub: final.append(sub)
    return final


def fix_eier_prefix(records):
    for rec in records:
        m = DIRECTIONAL.match(rec.get('eier_bruker', ''))
        if m:
            rec['brugets_navn'] = (rec['brugets_navn'] + ' ' + m.group(1)).strip()
            rec['eier_bruker'] = m.group(2)
    return records


def fix_gaards_sequence(records):
    gv = []
    for i, r in enumerate(records):
        if r['gaards_no']:
            try: gv.append((i, int(r['gaards_no'])))
            except ValueError: pass
    for k in range(1, len(gv)-1):
        idx, val = gv[k]
        _, pv = gv[k-1]; _, nv = gv[k+1]
        if pv < nv and not (pv < val <= nv):
            records[idx]['gaards_no'] = str(pv + 1)
    return records


def has_full_entry(d):
    return 'brugets_navn' in d and 'eier_bruker' in d


def merge_dict(target, source, keys=('brugets_navn', 'gaardens_navn', 'eier_bruker', 'skyld', 'anmerkn', 'postanstalt')):
    for k in keys:
        if k in source:
            target[k] = (target[k] + ' ' + source[k]).strip() if k in target else source[k]
    for k in ('gaards_no', 'brugs_no'):
        if k in source and k not in target: target[k] = source[k]


def build_record(d, cur_gaards, cur_gaardens):
    mark, ore, anm_from_skyld = parse_skyld(d['skyld']) if 'skyld' in d else ('', '', '')
    ore = ore.zfill(2) if ore and ore.isdigit() else ore
    anm = d.get('anmerkn', '')
    if anm_from_skyld and not anm: anm = anm_from_skyld
    elif anm_from_skyld and anm: anm = anm_from_skyld + ' ' + anm
    return {
        'gaards_no': cur_gaards, 'brugs_no': d.get('brugs_no', ''),
        'gaardens_navn': cur_gaardens,
        'brugets_navn': d.get('brugets_navn', ''),
        'eier_bruker': d.get('eier_bruker', ''),
        'mark': mark, 'ore': ore,
        'anmerkn': clean_anmerkn(anm),
        'postanstalt': clean_anmerkn(d.get('postanstalt', '')),
    }


def parse_page(pred, cached_cols=None, carry_g='', carry_gn='', carry_b=0):
    detected = detect_columns(pred)
    if not detected:
        if cached_cols: detected = cached_cols
        else: return None, None, None, cached_cols, ('', ''), carry_g, carry_gn, carry_b
    columns, bex, header_y = detected
    skyld_lo = columns[5][1]

    ovf_mark, ovf_ore, ove_mark, ove_ore = '', '', '', ''
    ove_anm, ove_post = '', ''
    anm_lo = columns[6][1]
    post_lo = columns[7][1]
    for line in pred.text_lines:
        if 'Overført' in line.text and 'Overføres' not in line.text:
            for l2 in pred.text_lines:
                if l2.bbox[0] > skyld_lo and abs(l2.bbox[1]-line.bbox[1]) < 30:
                    ovf_mark, ovf_ore, _ = parse_skyld(l2.text); break
        if 'Overføres' in line.text:
            ove_y = (line.bbox[1] + line.bbox[3]) / 2
            for l2 in pred.text_lines:
                if l2.bbox[0] > skyld_lo and abs(l2.bbox[1]-ove_y) < 30:
                    ove_mark, ove_ore, _ = parse_skyld(l2.text)
                    break
            for l2 in pred.text_lines:
                l2y = (l2.bbox[1] + l2.bbox[3]) / 2
                if l2.bbox[0] >= anm_lo and 0 < (ove_y - l2y) < 100:
                    t2 = l2.text.strip()
                    if DASH_RE.match(t2) or not t2: continue
                    if 'Overf' in t2: continue
                    if l2.bbox[0] >= post_lo: ove_post = (ove_post + ' ' + t2).strip()
                    else: ove_anm = (ove_anm + ' ' + t2).strip()

    sogn_anm, sogn_post = '', ''
    for line in pred.text_lines:
        if line.bbox[1] < header_y: continue
        if SOGN_RE.search(line.text):
            sogn_y = (line.bbox[1] + line.bbox[3]) / 2
            for l2 in pred.text_lines:
                l2y = (l2.bbox[1] + l2.bbox[3]) / 2
                if abs(l2y - sogn_y) > 20 or l2.bbox[0] < anm_lo: continue
                t2 = l2.text.strip()
                if DASH_RE.match(t2) or not t2: continue
                if l2.bbox[0] >= post_lo: sogn_post = (sogn_post + ' ' + t2).strip()
                else: sogn_anm = (sogn_anm + ' ' + t2).strip()
            break

    items = []
    for line in pred.text_lines:
        if line.bbox[1] < header_y or is_skip(line.text): continue
        if 'Overført' in line.text or 'Overføres' in line.text: continue
        t = line.text.strip()
        if DASH_RE.match(t): continue
        for col_name, val in process_line(list(line.bbox), t, columns, bex, skyld_lo):
            items.append((list(line.bbox), col_name, val))
    if not items: return None, None, None, detected, ('', '')

    rows = cluster_rows(items)
    records = []
    cur_gaards, cur_gaardens = carry_g, carry_gn
    last_brugs = carry_b
    pending = None

    def flush_pending():
        nonlocal pending, cur_gaards, cur_gaardens, last_brugs
        if not pending: return
        if 'brugs_no' not in pending:
            if 'gaards_no' in pending and pending['gaards_no'] != cur_gaards:
                pending['brugs_no'] = '1'; last_brugs = 0
            else: pending['brugs_no'] = str(last_brugs + 1)
        bn = pending['brugs_no']
        m_bn = re.match(r'^(\d+)', bn.strip())
        if m_bn: pending['brugs_no'] = m_bn.group(1)
        if 'gaards_no' in pending: cur_gaards = pending['gaards_no']
        if 'gaardens_navn' in pending: cur_gaardens = pending['gaardens_navn']
        try: last_brugs = int(pending['brugs_no'])
        except: pass
        records.append(build_record(pending, cur_gaards, cur_gaardens))
        pending = None

    for row in rows:
        d = {}
        for bbox, col, text in sorted(row, key=lambda x: x[0][0]):
            d[col] = (d[col] + ' ' + text) if col in d else text
        has_brugs = 'brugs_no' in d
        has_gaards = 'gaards_no' in d
        is_full = has_full_entry(d)

        if has_brugs:
            if pending:
                if has_full_entry(pending): flush_pending()
                else:
                    for k, v in pending.items():
                        if k not in d: d[k] = v
                    pending = None; has_gaards = 'gaards_no' in d
            if has_gaards: cur_gaards = d['gaards_no']; last_brugs = 0
            if 'gaardens_navn' in d: cur_gaardens = d['gaardens_navn']
            bn = d['brugs_no']
            m_bn = re.match(r'^(\d+)', bn.strip())
            if m_bn: d['brugs_no'] = m_bn.group(1)
            try: last_brugs = int(d['brugs_no'])
            except: pass
            records.append(build_record(d, cur_gaards, cur_gaardens))
        elif has_gaards or is_full:
            if pending and is_full and has_full_entry(pending): flush_pending()
            if pending: merge_dict(pending, d)
            else: pending = d
        else:
            if pending: merge_dict(pending, d)
            elif records:
                for key in ('brugets_navn', 'gaardens_navn', 'eier_bruker', 'anmerkn', 'postanstalt'):
                    if key in d: records[-1][key] = (records[-1].get(key,'') + ' ' + d[key]).strip()
                if 'gaardens_navn' in d: cur_gaardens = records[-1].get('gaardens_navn', cur_gaardens)

    flush_pending()
    if records and (ove_anm or ove_post):
        last = records[-1]
        if ove_anm: last['anmerkn'] = ((last.get('anmerkn','') or '') + ' ' + ove_anm).strip()
        if ove_post and not last.get('postanstalt'): last['postanstalt'] = ove_post
    records = fix_eier_prefix(records)
    records = fix_gaards_sequence(records)
    return records, (ovf_mark, ovf_ore), (ove_mark, ove_ore), detected, (sogn_anm, sogn_post), cur_gaards, cur_gaardens, last_brugs


EMPTY_REC = {'gaards_no':'','brugs_no':'','gaardens_navn':'','brugets_navn':'',
             'eier_bruker':'','mark':'','ore':'','anmerkn':'','postanstalt':''}

def make_df(records, ovf, ove, sogn=('', '')):
    ovf_row = {**EMPTY_REC, 'eier_bruker':'Overført','mark':ovf[0],'ore':ovf[1],
               'anmerkn':clean_anmerkn(sogn[0]),'postanstalt':clean_anmerkn(sogn[1])}
    ove_row = {**EMPTY_REC, 'eier_bruker':'Overføres','mark':ove[0],'ore':ove[1]}
    rows = [ovf_row] + records + [ove_row]
    df = pd.DataFrame(rows)
    df['brugs_no'] = df['brugs_no'].replace({'ō':'5','õ':'5'})
    for col in ('gaardens_navn','brugets_navn','eier_bruker'):
        df[col] = df[col].apply(clean)
    for col in ('anmerkn','postanstalt'):
        df[col] = df[col].apply(clean_anmerkn)
    for col in ('gaards_no','gaardens_navn'):
        df[col] = df[col].where(df[col] != df[col].shift(), '')
    return df


all_dfs = []
cached = None
cg, cgn, cb = '', '', 0
for i, pred in enumerate(all_predictions):
    records, ovf, ove, cached, sogn, cg, cgn, cb = parse_page(pred, cached, cg, cgn, cb)
    if not records:
        print(f'Page {i+1}: skipped (no headers or data found)')
        continue
    df = make_df(records, ovf, ove, sogn)
    all_dfs.append(df)
    print(f'Page {i+1}: {len(records)} rows')

if not all_dfs:
    print('ERROR: No pages parsed.')
else:
    all_df = pd.concat(all_dfs, ignore_index=True)

    # post-processing
    special = all_df['eier_bruker'].str.contains('Overført|Overføres', na=False)
    is_overføres = all_df['eier_bruker'].str.contains('Overføres', na=False)
    is_overført = all_df['eier_bruker'].str.contains('Overført', na=False) & ~is_overføres
    has_gaardens = all_df['gaardens_navn'].notna() & (all_df['gaardens_navn'].str.strip() != '') & ~special

    #gaards_no assignement
    pending_gaards = pd.NA
    for i in all_df.index:
        if special.iloc[i]: pending_gaards = pd.NA; continue
        g = all_df.at[i, 'gaards_no']
        if has_gaardens.iloc[i]:
            if pd.notna(pending_gaards):
                all_df.at[i, 'gaards_no'] = pending_gaards; pending_gaards = pd.NA
        else:
            if pd.notna(g):
                try: pending_gaards = int(float(g)); all_df.at[i, 'gaards_no'] = pd.NA
                except (ValueError, TypeError): all_df.at[i, 'gaards_no'] = pd.NA

    ng_idx = all_df.index[has_gaardens].tolist()
    for j, idx in enumerate(ng_idx):
        if pd.notna(all_df.at[idx, 'gaards_no']): continue
        for k in range(j - 1, -1, -1):
            if pd.notna(all_df.at[ng_idx[k], 'gaards_no']):
                try: all_df.at[idx, 'gaards_no'] = int(float(all_df.at[ng_idx[k], 'gaards_no'])) + (j - k)
                except (ValueError, TypeError): pass
                break

    # brugs_no fixes 
    cur_gaards, last_brugs = None, 0
    for i in all_df.index:
        if special.iloc[i]: continue
        g, b = all_df.at[i, 'gaards_no'], all_df.at[i, 'brugs_no']
        if has_gaardens.iloc[i] and pd.notna(g) and g != cur_gaards:
            cur_gaards = g; last_brugs = 0
        if pd.notna(b):
            try: bval = int(float(b))
            except (ValueError, TypeError): continue
            if bval == 0 or bval <= last_brugs:
                all_df.at[i, 'brugs_no'] = last_brugs + 1; last_brugs += 1
            else: last_brugs = bval
        elif not special.iloc[i] and pd.notna(all_df.at[i, 'eier_bruker']):
            last_brugs += 1; all_df.at[i, 'brugs_no'] = last_brugs

    # overfores and overfort 
    for i in all_df.index[is_overføres]:
        if pd.notna(all_df.at[i, 'mark']) and str(all_df.at[i, 'mark']).strip(): continue
        if i + 1 < len(all_df) and is_overført.iloc[i + 1]:
            all_df.at[i, 'mark'] = all_df.at[i + 1, 'mark']
            all_df.at[i, 'ore'] = all_df.at[i + 1, 'ore']
        else:
            prev_ovf = all_df.index[is_overført & (all_df.index < i)]
            start = prev_ovf.max() if len(prev_ovf) > 0 else 0
            base_m = int(all_df.at[start, 'mark'] or 0) if pd.notna(all_df.at[start, 'mark']) else 0
            base_o = int(all_df.at[start, 'ore'] or 0) if pd.notna(all_df.at[start, 'ore']) else 0
            data = all_df.iloc[start + 1:i][~special.iloc[start + 1:i]]
            sum_m = pd.to_numeric(data['mark'], errors='coerce').fillna(0).sum()
            sum_o = pd.to_numeric(data['ore'], errors='coerce').fillna(0).sum()
            total_ore = base_o + int(sum_o)
            all_df.at[i, 'mark'] = str(base_m + int(sum_m) + total_ore // 100)
            all_df.at[i, 'ore'] = f'{total_ore % 100:02d}'

    # zero in ore column 
    def pad_ore(v):
        if pd.isna(v) or str(v).strip() == '': return ''
        try: return f'{int(float(v)):02d}'
        except: return str(v)
    all_df['ore'] = all_df['ore'].apply(pad_ore)

    for i in all_df.index:
        if special.iloc[i]: continue
        m_val = str(all_df.at[i, 'mark']).strip() if pd.notna(all_df.at[i, 'mark']) else ''
        o_val = str(all_df.at[i, 'ore']).strip() if pd.notna(all_df.at[i, 'ore']) else ''
        if m_val and not o_val:
            try:
                mv = int(float(m_val))
                if 0 < mv <= 99:
                    all_df.at[i, 'ore'] = f'{mv:02d}'
                    all_df.at[i, 'mark'] = ''
            except (ValueError, TypeError): pass

    # anmerkn column 
    for i in all_df.index:
        if special.iloc[i] or i + 1 >= len(all_df): continue
        anm = str(all_df.at[i, 'anmerkn']) if pd.notna(all_df.at[i, 'anmerkn']) else ''
        m_leak = re.match(r'^(.*?)\s+(\d{1,2})\s+(\d{2})\s+(.*?)$', anm)
        if not m_leak: continue
        next_m = str(all_df.at[i+1, 'mark']).strip() if pd.notna(all_df.at[i+1, 'mark']) else ''
        next_o = str(all_df.at[i+1, 'ore']).strip() if pd.notna(all_df.at[i+1, 'ore']) else ''
        if not next_m and not next_o:
            all_df.at[i, 'anmerkn'] = (m_leak.group(1) + ' ' + m_leak.group(4)).strip()
            all_df.at[i+1, 'mark'] = m_leak.group(2)
            all_df.at[i+1, 'ore'] = m_leak.group(3).zfill(2)

    # clean gaardens 
    mask = all_df['gaardens_navn'].fillna('').str.match(r'^\d+$') & \
           (all_df['gaards_no'].isna() | (all_df['gaards_no'].astype(str).str.strip() == ''))
    all_df.loc[mask, 'gaardens_navn'] = ''

    # clean trailing dots 
    all_df['gaardens_navn'] = all_df['gaardens_navn'].fillna('').str.replace(
        r'[\s.]*\.{2,}[\s.\d]*$', '', regex=True).str.strip()

    # gaards no 
    for col in ('gaards_no', 'brugs_no'):
        all_df[col] = pd.to_numeric(all_df[col], errors='coerce').astype('Int64')

    all_df.to_csv('all_pages.csv', index=False, encoding='utf-8-sig')
    print(f'\nTotal: {len(all_df)} rows → all_pages.csv')




In [ ]:
# Cell 3: Post-processing → all_pages.csv
import pandas as pd, re

df = pd.read_csv('all_pages.csv', dtype=str).fillna('')
special = df['eier_bruker'].str.contains('Overført|Overføres', na=False)
DASH_RE = re.compile(r'^[\s\u2014\u2013\u2012\u2015—–-]+$')
DIRS = r'(?:søndre|nordre|østre|vestre|øvre|nedre|lille|store|mellem|skole-?gaard|skolejord|skole|med)'

# ── 1. Fix "directional...name" in eier_bruker ──
DOT_DIR = re.compile(r'^(' + DIRS + r')[\s.]{2,}(.+)$', re.I)
for i in df.index:
    if special.iloc[i]: continue
    ek = str(df.at[i, 'eier_bruker']).strip()
    m = DOT_DIR.match(ek)
    if m:
        df.at[i, 'brugets_navn'] = (str(df.at[i,'brugets_navn']).strip() + ' ' + m.group(1)).strip()
        df.at[i, 'eier_bruker'] = re.sub(r'^[\s.]+', '', m.group(2)).strip()

DOT_DIR2 = re.compile(r'^(' + DIRS + r')\.\s+(.+)$', re.I)
for i in df.index:
    if special.iloc[i]: continue
    ek = str(df.at[i, 'eier_bruker']).strip()
    m = DOT_DIR2.match(ek)
    if m:
        df.at[i, 'brugets_navn'] = (str(df.at[i,'brugets_navn']).strip() + ' ' + m.group(1)).strip()
        df.at[i, 'eier_bruker'] = m.group(2).strip()

SKOG_RE = re.compile(r'^\(skog\)\s+(.+)$', re.I)
for i in df.index:
    if special.iloc[i]: continue
    m = SKOG_RE.match(str(df.at[i, 'eier_bruker']).strip())
    if m:
        df.at[i, 'brugets_navn'] = (str(df.at[i,'brugets_navn']).strip() + ' (skog)').strip()
        df.at[i, 'eier_bruker'] = m.group(1).strip()

# ── 1b. Fix hyphenated prefix in eier_bruker (e.g. "Dam- Ole Mellegaard") ──
HYPH_PREFIX = re.compile(r'^(\w+)-\s+(.+)$')
for i in df.index:
    if special.iloc[i]: continue
    ek = str(df.at[i, 'eier_bruker']).strip()
    m = HYPH_PREFIX.match(ek)
    if m:
        prefix = m.group(1)
        # only if prefix looks like a place-name fragment (not a real first name)
        if len(prefix) <= 6 or prefix[0].isupper():
            bn = str(df.at[i, 'brugets_navn']).strip()
            df.at[i, 'brugets_navn'] = bn + ' ' + prefix + '-' if bn else prefix + '-'
            df.at[i, 'eier_bruker'] = m.group(2).strip()

# ── 2. Fix directional prefix in brugets_navn ──
DIR_PREFIX = re.compile(r'^(' + DIRS + r')[\s.]+(\w.+)$', re.I)
for i in df.index:
    if special.iloc[i]: continue
    bn = str(df.at[i, 'brugets_navn']).strip()
    gn = str(df.at[i, 'gaardens_navn']).strip()
    m = DIR_PREFIX.match(bn)
    if m and not gn:
        df.at[i, 'brugets_navn'] = m.group(2).strip()

# ── 3. Clean dots from text columns ──
for col in ('gaardens_navn','brugets_navn','eier_bruker'):
    df[col] = df[col].apply(lambda x: re.sub(r'\.{2,}', ' ', str(x)).strip())
    df[col] = df[col].apply(lambda x: re.sub(r'\s{2,}', ' ', str(x)).strip())
    df[col] = df[col].apply(lambda x: re.sub(r'\s*\.\s*$', '', str(x)).strip())
    df[col] = df[col].apply(lambda x: re.sub(r'^\.\s+', '', str(x)).strip())

# ── 4. Merge phantom rows (gaards_no + gaardens_navn but no real data) ──
BARE_DIR = re.compile(r'^' + DIRS + r'$', re.I)
special = df['eier_bruker'].str.contains('Overført|Overføres', na=False)
to_drop = []
phantom_brugs = {}  # gaards_no → last brugs_no before phantom, for step 5
for i in df.index:
    if special.iloc[i]: continue
    bn = str(df.at[i, 'brugets_navn']).strip()
    ek = str(df.at[i, 'eier_bruker']).strip()
    gn = str(df.at[i, 'gaardens_navn']).strip()
    g = str(df.at[i, 'gaards_no']).strip()
    mv = str(df.at[i, 'mark']).strip()
    ov = str(df.at[i, 'ore']).strip()
    is_phantom = gn and not ek and not mv and (not ov or ov in ('','nan'))
    is_phantom = is_phantom and (not bn or BARE_DIR.match(bn))
    if not is_phantom: continue
    # find the last real brugs_no before this phantom for this gaards
    if g and g not in ('','nan'):
        for k in range(i-1, max(0, i-50), -1):
            if special.iloc[k]: break
            kg = str(df.at[k, 'gaards_no']).strip()
            kb = str(df.at[k, 'brugs_no']).strip()
            if kg == g and kb and kb not in ('','nan'):
                try: phantom_brugs[g] = int(float(kb))
                except: pass
                break
    for j in range(i+1, min(i+5, len(df))):
        if special.iloc[j]: break
        jbn = str(df.at[j, 'brugets_navn']).strip()
        jek = str(df.at[j, 'eier_bruker']).strip()
        if jbn or jek:
            if g and g not in ('','nan'):
                jg = str(df.at[j, 'gaards_no']).strip()
                if not jg or jg in ('','nan'): df.at[j, 'gaards_no'] = g
            jgn = str(df.at[j, 'gaardens_navn']).strip()
            merged_gn = (gn + ' ' + bn).strip() if bn and BARE_DIR.match(bn) else gn
            if not jgn or jgn in ('','nan'): df.at[j, 'gaardens_navn'] = merged_gn
            to_drop.append(i)
            break
if to_drop:
    df = df.drop(to_drop).reset_index(drop=True)
    special = df['eier_bruker'].str.contains('Overført|Overføres', na=False)

# ── 4b. Merge word-fragment continuation rows ──
# Catches: "sted" alone (OCR line break), "nedre" + "og Hans..." (multi-line entry)
special = df['eier_bruker'].str.contains('Overført|Overføres', na=False)
BARE_FRAG = re.compile(r'^(?:' + DIRS + r')$', re.I)
to_drop2 = []
for i in range(1, len(df)):
    if special.iloc[i] or special.iloc[i-1]: continue
    bn = str(df.at[i, 'brugets_navn']).strip()
    ek = str(df.at[i, 'eier_bruker']).strip()
    gn = str(df.at[i, 'gaardens_navn']).strip()
    g = str(df.at[i, 'gaards_no']).strip()
    mv = str(df.at[i, 'mark']).strip()
    ov = str(df.at[i, 'ore']).strip()
    no_nums = (not mv or mv in ('','nan')) and (not ov or ov in ('','nan'))
    no_id = (not g or g in ('','nan')) and (not gn or gn in ('','nan'))
    # case 1: pure fragment (short lowercase, no eier)
    is_frag = no_id and no_nums and bn and not ek and len(bn) <= 12 and bn[0].islower()
    # case 2: directional bn + continuation eier starting with "og "
    is_cont = no_id and no_nums and bn and BARE_FRAG.match(bn) and ek.startswith('og ')
    if not is_frag and not is_cont: continue
    prev_bn = str(df.at[i-1, 'brugets_navn']).strip()
    prev_ek = str(df.at[i-1, 'eier_bruker']).strip()
    if is_frag:
        if prev_bn:
            sep = '' if prev_bn.endswith('-') else ' '
            if prev_bn.endswith('-'): prev_bn = prev_bn[:-1]
            df.at[i-1, 'brugets_navn'] = prev_bn + sep + bn
            to_drop2.append(i)
        elif prev_ek:
            sep = '' if prev_ek.endswith('-') else ' '
            if prev_ek.endswith('-'): prev_ek = prev_ek[:-1]
            df.at[i-1, 'eier_bruker'] = prev_ek + sep + bn
            to_drop2.append(i)
    elif is_cont:
        df.at[i-1, 'brugets_navn'] = (prev_bn + ' ' + bn).strip()
        df.at[i-1, 'eier_bruker'] = (prev_ek + ' ' + ek).strip() if prev_ek else ek
        to_drop2.append(i)
if to_drop2:
    df = df.drop(to_drop2).reset_index(drop=True)
    special = df['eier_bruker'].str.contains('Overført|Overføres', na=False)

# ── 4c. Remove empty rows and close brugs_no gaps they leave behind ──
special = df['eier_bruker'].str.contains('Overført|Overføres', na=False)
empty_mask = ~special & df[['brugets_navn','eier_bruker','mark','ore']].apply(
    lambda r: all(str(v).strip() in ('','nan') for v in r), axis=1)
if empty_mask.any():
    # record which (gaards_no, brugs_no) slots are being removed
    removed_slots = set()
    for i in df.index[empty_mask]:
        g = str(df.at[i, 'gaards_no']).strip()
        b = str(df.at[i, 'brugs_no']).strip()
        if not g or g in ('','nan'):
            # inherit gaards from nearest previous
            for k in range(i-1, max(0, i-20), -1):
                kg = str(df.at[k, 'gaards_no']).strip()
                if kg and kg not in ('','nan'): g = kg; break
        if g and b and b not in ('','nan'):
            try: removed_slots.add((g, int(float(b))))
            except: pass
    df = df[~empty_mask].reset_index(drop=True)
    special = df['eier_bruker'].str.contains('Overført|Overføres', na=False)
    # for each gaards that had removals, shift brugs_no down to close gaps
    affected_gaards = {g for g, _ in removed_slots}
    for ag in affected_gaards:
        removed_in_g = sorted(b for g, b in removed_slots if g == ag)
        cur_g_mask = []
        active = False
        for i in df.index:
            if special.iloc[i]: active = False; continue
            g = str(df.at[i, 'gaards_no']).strip()
            if g == ag: active = True
            elif g and g not in ('','nan') and g != ag: active = False
            if active:
                cur_g_mask.append(i)
        for i in cur_g_mask:
            b = str(df.at[i, 'brugs_no']).strip()
            if not b or b in ('','nan'): continue
            try:
                bv = int(float(b))
                shift = sum(1 for rb in removed_in_g if rb < bv)
                if shift > 0:
                    df.at[i, 'brugs_no'] = str(bv - shift)
            except: pass

# ── 5. Fix brugs_no: trust parsed values, only fill where missing ──
cur_g, last_b = None, 0
gs = dict(phantom_brugs)  # seed with pre-phantom state for cross-page continuity
for i in df.index:
    if special.iloc[i]: continue
    g = str(df.at[i, 'gaards_no']).strip()
    gn = str(df.at[i, 'gaardens_navn']).strip()
    if g and g not in ('','nan') and g != str(cur_g):
        cur_g = g; last_b = gs.get(g, 0)
    b = str(df.at[i, 'brugs_no']).strip()
    has_c = str(df.at[i, 'eier_bruker']).strip() or str(df.at[i, 'brugets_navn']).strip()
    if b and b not in ('','nan'):
        try:
            bv = int(float(b))
            if bv <= last_b:
                # OCR misread or duplicate — increment instead
                last_b += 1
                df.at[i, 'brugs_no'] = str(last_b)
            else:
                last_b = bv
        except: pass
    elif has_c:
        last_b += 1
        df.at[i, 'brugs_no'] = str(last_b)
    if cur_g: gs[cur_g] = last_b

# ── 5b. Fix brugs_no concatenation artifacts (e.g. 2,23,24,25 → 2,3,4,5) ──
# Per gaards block: detect a big jump, compute the offset, fix entire tail
cur_g = None
block_start = 0
block_indices = []
def fix_block(indices):
    if len(indices) < 2: return
    vals = []
    for idx in indices:
        b = str(df.at[idx, 'brugs_no']).strip()
        try: vals.append((idx, int(float(b))))
        except: vals.append((idx, None))
    for k in range(1, len(vals)):
        if vals[k][1] is None or vals[k-1][1] is None: continue
        prev, cur = vals[k-1][1], vals[k][1]
        if cur > prev + 10:
            # found a big jump — compute offset
            sp = str(prev)
            sc = str(cur)
            if sc.startswith(sp):
                offset = cur - int(sc[len(sp):])
                # apply offset to this row and all subsequent in block
                for m in range(k, len(vals)):
                    if vals[m][1] is not None:
                        fixed = vals[m][1] - offset
                        if fixed > 0:
                            df.at[vals[m][0], 'brugs_no'] = str(fixed)
                            vals[m] = (vals[m][0], fixed)
                break

for i in df.index:
    if special.iloc[i]:
        fix_block(block_indices); block_indices = []; cur_g = None
        continue
    g = str(df.at[i, 'gaards_no']).strip()
    if g and g not in ('','nan') and g != str(cur_g):
        fix_block(block_indices)
        cur_g = g; block_indices = []
    b = str(df.at[i, 'brugs_no']).strip()
    if b and b not in ('','nan'):
        block_indices.append(i)
fix_block(block_indices)

# ── 6. Fix gaards_no forward-fill ──
has_gn = df['gaardens_navn'].str.strip().ne('') & ~special
last_g = None
for i in df.index:
    if special.iloc[i]: continue
    g = str(df.at[i, 'gaards_no']).strip()
    if g and g not in ('','nan'):
        try: last_g = int(float(g))
        except: pass
    elif has_gn.iloc[i] and last_g is not None:
        last_g += 1; df.at[i, 'gaards_no'] = str(last_g)

# ── 7. Fix mark/ore swap ──
for i in df.index:
    if special.iloc[i]: continue
    mv = str(df.at[i, 'mark']).strip()
    ov = str(df.at[i, 'ore']).strip()
    if mv and mv not in ('','nan') and (not ov or ov in ('','nan')):
        try:
            val = int(float(mv))
            if 0 < val <= 99: df.at[i, 'ore'] = f'{val:02d}'; df.at[i, 'mark'] = ''
        except: pass

# ── 8. Fix Overføres missing values ──
is_ovf = df['eier_bruker'].str.contains('Overført', na=False) & ~df['eier_bruker'].str.contains('Overføres', na=False)
is_ove = df['eier_bruker'].str.contains('Overføres', na=False)
for i in df.index[is_ove]:
    mv = str(df.at[i, 'mark']).strip()
    if mv and mv not in ('','nan'): continue
    # try copying from next Overført row
    if i+1 < len(df) and is_ovf.iloc[i+1]:
        nm = str(df.at[i+1, 'mark']).strip()
        if nm and nm not in ('','nan'):
            df.at[i, 'mark'] = nm; df.at[i, 'ore'] = df.at[i+1, 'ore']
            continue
    # fallback: calculate from previous Overført + sum of data rows
    prev_ovf_idx = df.index[is_ovf & (df.index < i)]
    start = prev_ovf_idx.max() if len(prev_ovf_idx) > 0 else 0
    base_m = 0; base_o = 0
    sm = str(df.at[start, 'mark']).strip() if start > 0 else ''
    so = str(df.at[start, 'ore']).strip() if start > 0 else ''
    try: base_m = int(float(sm)) if sm and sm not in ('','nan') else 0
    except: pass
    try: base_o = int(float(so)) if so and so not in ('','nan') else 0
    except: pass
    data = df.iloc[start+1:i]
    data = data[~data['eier_bruker'].str.contains('Overført|Overføres', na=False)]
    sum_m = pd.to_numeric(data['mark'], errors='coerce').fillna(0).sum()
    sum_o = pd.to_numeric(data['ore'], errors='coerce').fillna(0).sum()
    total_o = base_o + int(sum_o)
    total_m = base_m + int(sum_m) + total_o // 100
    df.at[i, 'mark'] = str(total_m)
    df.at[i, 'ore'] = f'{total_o % 100:02d}'
    df.at[i, 'anmerkn'] = 'BEREGNET (ikke fra OCR)'
    # also fix the following Overført if empty
    if i+1 < len(df) and is_ovf.iloc[i+1]:
        df.at[i+1, 'mark'] = df.at[i, 'mark']
        df.at[i+1, 'ore'] = df.at[i, 'ore']
        df.at[i+1, 'anmerkn'] = 'BEREGNET (ikke fra OCR)'

# ── 8b. Move Overført row annotations to first data row after it ──
is_ovf2 = df['eier_bruker'].str.strip() == 'Overført'
for i in df.index[is_ovf2]:
    a = str(df.at[i, 'anmerkn']).strip()
    if a and 'BEREGNET' not in a:
        # find next non-special row
        for j in range(i+1, min(i+5, len(df))):
            if not special.iloc[j]:
                cur_a = str(df.at[j, 'anmerkn']).strip()
                if not cur_a:
                    df.at[j, 'anmerkn'] = a
                df.at[i, 'anmerkn'] = ''
                break

# ── 9. Clean anmerkn ──
MERGE_TAILS = re.compile(r'(skr\.?|Udt\.?|kaldet|ogsaa|Alm\.?)$', re.I)
ANN_PREFIXES = re.compile(r'^(Udt\.?|Alm\.?\s*skr\.?|Ogsaa|kaldet)$', re.I)
for i in df.index:
    if special.iloc[i]: continue
    anm = str(df.at[i, 'anmerkn']).strip()
    anm = re.sub(r'(\w)-\d+\s+\d+\s+(\w)', r'\1\2', anm)
    if anm and not re.match(r'^\d', anm):
        anm = re.sub(r',?\s*\d{1,3}\s*$', '', anm).strip()
    df.at[i, 'anmerkn'] = anm

special = df['eier_bruker'].str.contains('Overført|Overføres', na=False)
anm = df['anmerkn'].astype(str).tolist()
# pass 1: merge when current ends with truncation marker
for i in range(len(anm)-1):
    if special.iloc[i] or special.iloc[i+1]: continue
    a, b = anm[i].strip(), anm[i+1].strip()
    if a and b and (MERGE_TAILS.search(a) or a.endswith('-')):
        anm[i] = a.rstrip() + ('' if a.endswith('-') else ' ') + b; anm[i+1] = ''
# pass 2: merge short lowercase fragments
for i in range(len(anm)-1):
    if special.iloc[i] or special.iloc[i+1]: continue
    a, b = anm[i].strip(), anm[i+1].strip()
    if a and b and len(b) < 15 and b[0].islower():
        anm[i] = a + b; anm[i+1] = ''
# pass 3: merge short capitalized place names after "Udt." or "skr." endings
for i in range(len(anm)-1):
    if special.iloc[i] or special.iloc[i+1]: continue
    a, b = anm[i].strip(), anm[i+1].strip()
    if not a or not b: continue
    if MERGE_TAILS.search(a) and len(b) < 20 and re.match(r'^[A-ZÆØÅ]', b):
        anm[i] = a + ' ' + b; anm[i+1] = ''
df['anmerkn'] = anm

# fix duplicated suffixes: "Haslerud.rud." → "Haslerud."
df['anmerkn'] = df['anmerkn'].apply(
    lambda x: re.sub(r'(\w{2,})\.\1\.', r'\1.', str(x)).strip())

# fix leaked postanstalt in middle of anmerkn
# pattern: "Udt. Kaalaa-Trøgstad.baann." → remove known post names from middle
for i in df.index:
    if special.iloc[i]: continue
    a = str(df.at[i, 'anmerkn']).strip()
    if not a: continue
    # remove known post office names embedded mid-annotation
    a = re.sub(r'[-.\s](?:Trøgstad|Slitu|Mysen|Frøshaug?)\.?\s*', '', a).strip()
    # clean double dots and leading dots/spaces
    a = re.sub(r'\.\.+', '.', a)
    a = re.sub(r'^\.\s*', '', a).strip()
    df.at[i, 'anmerkn'] = a
df['anmerkn'] = df['anmerkn'].apply(lambda x: re.sub(r'-\s+', '-', str(x)).strip())
df['anmerkn'] = df['anmerkn'].apply(lambda x: re.sub(r'([A-Za-zÆØÅæøå])-([a-zæøå])', r'\1\2', str(x)).strip())

for i in df.index:
    p = str(df.at[i, 'postanstalt']).strip()
    if p and ANN_PREFIXES.match(p):
        a = str(df.at[i, 'anmerkn']).strip()
        df.at[i, 'anmerkn'] = (a+' '+p).strip() if a else p; df.at[i, 'postanstalt'] = ''
KNOWN_POST = {'trøgstad','slitu','mysen','frøshaug','frøsau','frøsan'}
for v in df['postanstalt']:
    v = str(v).strip().rstrip('.')
    if v and len(v) > 1: KNOWN_POST.add(v.lower())
for i in df.index:
    a = str(df.at[i, 'anmerkn']).strip(); p = str(df.at[i, 'postanstalt']).strip()
    if not a or p: continue
    for kp in sorted(KNOWN_POST, key=len, reverse=True):
        m = re.search(r'[\s-]+(' + re.escape(kp) + r'\.?)$', a, re.I)
        if m: df.at[i, 'postanstalt'] = m.group(1); df.at[i, 'anmerkn'] = a[:m.start()].strip(); break
anm = df['anmerkn'].tolist()
for i in range(1, len(anm)):
    if special.iloc[i]: continue
    cur, prev = str(anm[i]).strip(), str(anm[i-1]).strip()
    if cur and prev and len(cur) < 15 and prev.endswith(cur): anm[i] = ''
df['anmerkn'] = anm

# move standalone place names from anmerkn to postanstalt
for i in df.index:
    if special.iloc[i]: continue
    a = str(df.at[i, 'anmerkn']).strip()
    p = str(df.at[i, 'postanstalt']).strip()
    if a and not p and not re.match(r'^(Udt|Alm|Ogsaa|Kaldes|kaldet)', a, re.I):
        al = a.rstrip('.').lower()
        if al in KNOWN_POST:
            df.at[i, 'postanstalt'] = a
            df.at[i, 'anmerkn'] = ''

# ── 9b. Clean trailing dots and reconstruct hyphen fragments in brugets_navn ──
for i in df.index:
    if special.iloc[i]: continue
    bn = str(df.at[i, 'brugets_navn']).strip()
    # remove trailing dots: "Skrikerud nordre ." → "Skrikerud nordre"
    bn = re.sub(r'\s*\.+\s*$', '', bn).strip()
    # reconstruct hyphen fragments: "Tullebæk og merud Dam-" → "Tullebæk og Dammerud"
    DIR_WORDS = {'søndre','nordre','østre','vestre','øvre','nedre','lille','store','mellem'}
    m = re.match(r'^(.+?)\s+(\w+)\s+(\w+)\s+(\w+)-$', bn)
    if m and m.group(3).lower() in DIR_WORDS:
        # "base fragment dir Prefix-" → "base Prefix+fragment dir"
        base, frag, dirw, prefix = m.group(1), m.group(2), m.group(3), m.group(4)
        bn = base + ' ' + prefix + frag + ' ' + dirw
    else:
        m = re.match(r'^(.+?)\s+(\w+)\s+(\w+)-$', bn)
        if m:
            base, frag, prefix = m.group(1), m.group(2), m.group(3)
            bn = base + ' ' + prefix + frag
    df.at[i, 'brugets_navn'] = bn

# ── 10. Clean artifacts ──
df['brugets_navn'] = df['brugets_navn'].apply(lambda x: re.sub(r'\s*[\(\[]?[A-Z]{5,}[\)\]]?', '', str(x)).strip())
# remove OCR garbage in parentheses (e.g. "(Sycartoreal")
df['brugets_navn'] = df['brugets_navn'].apply(lambda x: re.sub(r'\s*\([^)]*$', '', str(x)).strip())
df['brugets_navn'] = df['brugets_navn'].apply(lambda x: re.sub(r'\s*\([A-Z][a-z]*[^)]{5,}\)', '', str(x)).strip())
df['brugets_navn'] = df['brugets_navn'].apply(lambda x: re.sub(r'\s+\d{6,}', '', str(x)).strip())
for col in ('anmerkn','postanstalt'):
    df[col] = df[col].apply(lambda x: '' if not str(x).strip() or DASH_RE.match(str(x).strip()) else str(x).strip())
df['postanstalt'] = df['postanstalt'].apply(lambda x: '' if re.match(r'^\d{1,2}$', str(x).strip()) else str(x).strip())
df['gaardens_navn'] = df['gaardens_navn'].fillna('').str.replace(r'[\s.]*\.{2,}[\s.\d]*$', '', regex=True).str.strip()
mask = df['gaardens_navn'].str.match(r'^\d+$') & df['gaards_no'].astype(str).str.strip().isin(['','nan'])
df.loc[mask, 'gaardens_navn'] = ''

# ── 11. Deduplicate display ──
special = df['eier_bruker'].str.contains('Overført|Overføres', na=False)
for col in ('gaards_no','gaardens_navn'):
    prev = None
    for i in df.index:
        if special.iloc[i]: prev = None; continue
        val = str(df.at[i, col]).strip()
        if val and val != 'nan':
            if val == str(prev).strip(): df.at[i, col] = ''
            else: prev = val

# ── 12. Types & zero-pad ──
for col in ('gaards_no','brugs_no'):
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
# ensure ore is always zero-padded string (prevent Excel from stripping leading zeros)
df['ore'] = df['ore'].astype(str).apply(lambda v: f'{int(float(v)):02d}'
    if v.strip() not in ('','nan') and re.match(r'^\d+\.?\d*$', v.strip())
    else ('' if v.strip() in ('','nan') else v.strip()))
df['mark'] = df['mark'].astype(str).apply(lambda v: '' if v.strip() in ('','nan') else v.strip())

df.to_csv('all_pages.csv', index=False, encoding='utf-8-sig')
print(f'Done: {len(df)} rows → all_pages.csv')

# ── 13. Extract summary pages (herred overview tables) ──
DOTGAP = r'[\s.]{3,}'  # matches ". . . ." or "....." or ". .. ."
try:
    for i, pred in enumerate(all_predictions):
        text = ' '.join(ln.text for ln in pred.text_lines)
        if 'matrikelskyld' not in text.lower() or 'matrikulerede' not in text.lower():
            continue
        lines = sorted(pred.text_lines, key=lambda l: l.bbox[1])
        data = []
        for ln in lines:
            t = ln.text.strip()
            if 'herred' in t.lower() and len(t) < 50 and not any(c.isdigit() for c in t):
                data.append(('herred', t.rstrip('.')))
                break
        for ln in lines:
            t = ln.text.strip()
            # "Samlet matrikelskyld: Sogn . . . 1662 mark 47 øre"
            m = re.search(r'matrikelskyld\s*[.:]\s*(.+?)' + DOTGAP + r'(\d{3,5})\s+mark\s+(\d{1,2})\s+øre', t, re.I)
            if m:
                data.append(('matrikelskyld_' + m.group(1).strip(), f'{m.group(2)} mark {m.group(3)} øre'))
                continue
            # "Samlet matrikelskyld . . . 1440 mark 64 øre"
            m = re.search(r'matrikelskyld' + DOTGAP + r'(\d{3,5})\s+mark\s+(\d{1,2})\s+øre', t, re.I)
            if m:
                data.append(('matrikelskyld', f'{m.group(1)} mark {m.group(2)} øre'))
                continue
            # total: "2454 mark 09 øre" (standalone)
            m = re.match(r'^\s*(\d{3,5})\s+mark\s+(\d{1,2})\s+øre\s*$', t)
            if m:
                data.append(('matrikelskyld_total', f'{m.group(1)} mark {m.group(2)} øre'))
                continue
            # subdivision: "Trøgstad sogn . . . 1662 mark 47 øre"
            m = re.search(r'(\w[\w\s]*?)\s+(?:sogn)\s*' + DOTGAP + r'(\d{3,5})\s+mark\s+(\d{1,2})\s+øre', t)
            if m:
                data.append(('matrikelskyld_' + m.group(1).strip() + ' sogn', f'{m.group(2)} mark {m.group(3)} øre'))
                continue
            # "Baastad » . . 791 » 62 »"
            m = re.search(r'(\w+)\s+»[\s.]*(\d{2,4})\s*»?\s*(\d{1,2})\s*»', t)
            if m:
                data.append(('matrikelskyld_' + m.group(1).strip(), f'{m.group(2)} mark {m.group(3)} øre'))
                continue
            # "Antal matrikulerede brug: Sogn . . . 441"
            m = re.search(r'matrikulerede\s+brug\s*[.:]\s*(.+?)' + DOTGAP + r'(\d{2,4})\s*$', t, re.I)
            if m:
                data.append(('antal_brug_' + m.group(1).strip(), m.group(2)))
                continue
            # "Antal matrikulerede brug . . . 574"
            m = re.search(r'matrikulerede\s+brug' + DOTGAP + r'(\d{2,4})\s*$', t, re.I)
            if m:
                data.append(('antal_brug', m.group(1)))
                continue
            # "Baastad » . . . 326"
            m = re.search(r'(\w+)\s+»[\s.]*(\d{2,4})\s*$', t)
            if m and 'matrikel' not in t.lower() and 'mark' not in t.lower():
                data.append(('antal_brug_' + m.group(1).strip(), m.group(2)))
                continue
            # standalone total
            if re.match(r'^\s*\d{3,4}\s*$', t):
                data.append(('total', t.strip()))
                continue
            # "Fladeindhold . . . 67.64 km²" or "Samlet fladeindhold . . . 205.73 km.²"
            m = re.search(r'[Ff]ladeindhold[\s.]*?([\d.]+)\s*km', t)
            if m:
                data.append(('fladeindhold', m.group(1) + ' km²'))
                continue
            # folkemængde (may span 2 lines)
            if 'folkemængde' in t.lower() or 'folketælling' in t.lower():
                m = re.search(r'(\d{3,5})\s*$', t)
                if m: data.append(('folkemængde_3_dec_1900', m.group(1)))
                continue
            if 'december' in t.lower() and '1900' in t:
                m = re.search(r'(\d{3,5})\s*$', t)
                if m: data.append(('folkemængde_3_dec_1900', m.group(1)))
                continue
        if data:
            sdf = pd.DataFrame(data, columns=['field', 'value'])
            fname = f'summary_page_{i+1}.csv'
            sdf.to_csv(fname, index=False, encoding='utf-8-sig')
            herred = next((v for f, v in data if f == 'herred'), f'page {i+1}')
            print(f'Summary ({herred}) → {fname}:')
            for _, row in sdf.iterrows():
                print(f'  {row["field"]:40s} {row["value"]}')
except Exception as e:
    print(f'Summary extraction failed: {e}')







